# SVCM — ITI-95 — Récupération ValueSet

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-95
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12206&cid=48518

Récupérer le ValueSet `JDV-J245-Civilite-CISIS` par URL canonique, puis par `fullUrl`.

## Configuration

In [ ]:
import requests
import json
import time
import os
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_BASE  = "https://smt.esante.gouv.fr/api"
WP_BASE   = "https://smt.esante.gouv.fr/wp-json/ans"
API_KEY   = "ZiaoDxF8Je0CfBNzu..."   # Remplacer par la clé API complète

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "X-API-KEY": API_KEY,
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "02_iti95_valueset"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


---
## Étapes 0–30 — Récupération du ValueSet JDV-J245-Civilite-CISIS

**Requête** : `GET /fhir/ValueSet?url=https://mos.esante.gouv.fr/NOS/JDV_J245-Civilite-CISIS/FHIR/JDV-J245-Civilite-CISIS&_format=json`

In [ ]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET ValueSet par URL canonique
JDV_J245_URL = "https://mos.esante.gouv.fr/NOS/JDV_J245-Civilite-CISIS/FHIR/JDV-J245-Civilite-CISIS"
url_search = f"{FHIR_BASE}/ValueSet?url={quote(JDV_J245_URL)}&_format=json"

r_search = http_get(url_search)
bundle = r_search.json()

# Étape 20 — Réception et intégration
entries = bundle.get("entry", [])
assert len(entries) > 0, f"Aucune entrée trouvée pour {JDV_J245_URL}"

full_url   = entries[0].get("fullUrl", "")
vs_summary = entries[0].get("resource", {})

print(f"Entrées trouvées : {len(entries)}")
print(f"fullUrl          : {full_url}")
print(f"  name    : {vs_summary.get('name')}")
print(f"  version : {vs_summary.get('version')}")
print(f"  status  : {vs_summary.get('status')}")

# Étape 30 — PREUVE : GET par fullUrl
print(f"\n[PREUVE étape 30] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
vs = r_direct.json()
save_artifact(30, "jdv-j245-civilite-cisis.json", vs)

print_keys(vs, "resourceType", "id", "url", "version", "name", "title", "status")
includes = vs.get("compose", {}).get("include", [])
total_concepts = sum(len(inc.get("concept", [])) for inc in includes)
print(f"\nNombre de concepts : {total_concepts}")
for inc in includes:
    for c in inc.get("concept", []):
        print(f"  {c.get('code')} — {c.get('display')}")
